# Gold Layer: Investment Scores

Produces one row per listing with a composite investment score across seven normalised signals.
Two signals (location quality, rental yield) are stubs pending confirmation of additional Silver tables.

**Depends on:** `GOLD.BOROUGH_SUMMARY` (run `gold_borough_summary.ipynb` first)

**Output:** `AIRBNB_INVESTMENT_DB.GOLD.INVESTMENT_SCORES`

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
DB         = 'AIRBNB_INVESTMENT_DB'
SCHEMA_SIL = 'SILVER'
SCHEMA_GLD = 'GOLD'
TABLE_OUT  = 'INVESTMENT_SCORES'

NORM_LOCATION_STUB = 0.5
# TODO: replace with real transport score from BRISTOL_TRANSPORT_SILVER

NORM_YIELD_STUB = 0.5
# TODO: replace with real yield from BRISTOL_PROPERTY_PRICE_SILVER

SCORE_CONFIDENCE = 'medium'
# TODO: upgrade to 'high' once both stubs replaced with real data

In [ ]:
def minmax_norm(series, invert=False):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([0.5] * len(series), index=series.index)
    normed = (series - min_val) / (max_val - min_val)
    return 1 - normed if invert else normed


def check_borough_summary(session):
    result = session.sql('''
        SELECT COUNT(*) AS tbl_count
        FROM AIRBNB_INVESTMENT_DB.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = 'GOLD'
          AND TABLE_NAME = 'BOROUGH_SUMMARY'
    ''').to_pandas()
    if result['TBL_COUNT'][0] == 0:
        raise Exception(
            'BOROUGH_SUMMARY not found. '
            'Run gold_borough_summary.ipynb first.'
        )
    print('BOROUGH_SUMMARY confirmed — proceeding.')


def load_silver(session):
    df = session.sql('''
        SELECT id, listing_url, name, host_id, neighbourhood_cleansed,
               property_type, room_type, accommodates, bathrooms, bedrooms,
               price, estimated_occupancy, estimated_revenue,
               number_of_reviews, review_scores_rating,
               review_scores_cleanliness, review_scores_location,
               review_scores_value
        FROM AIRBNB_INVESTMENT_DB.SILVER.BRISTOL_LISTINGS_SILVER
    ''').to_pandas()
    print(f'Loaded {len(df)} listings from Silver')
    return df


def compute_investment_scores(df):
    df = df.copy()

    df['norm_price']      = minmax_norm(df['price'], invert=True)
    df['norm_occupancy']  = minmax_norm(df['estimated_occupancy'])
    df['norm_revenue']    = minmax_norm(df['estimated_revenue'])
    df['norm_rating']     = minmax_norm(df['review_scores_rating'])
    df['norm_review_vol'] = minmax_norm(df['number_of_reviews'])
    df['norm_location']   = NORM_LOCATION_STUB
    df['norm_yield']      = NORM_YIELD_STUB

    df['investment_score'] = (
        (df['norm_occupancy']  * 0.20 +
         df['norm_revenue']    * 0.20 +
         df['norm_rating']     * 0.15 +
         df['norm_price']      * 0.15 +
         df['norm_review_vol'] * 0.10 +
         df['norm_location']   * 0.10 +
         df['norm_yield']      * 0.10) * 100
    ).round(2)

    df['score_confidence'] = SCORE_CONFIDENCE
    df['computed_at']      = pd.Timestamp.now()

    print(f'Investment scores computed for {len(df)} listings')
    return df


def write_gold(session, df):
    session.write_pandas(
        df,
        'INVESTMENT_SCORES',
        database='AIRBNB_INVESTMENT_DB',
        schema='GOLD',
        overwrite=True
    )
    print(f'Written {len(df)} rows to GOLD.INVESTMENT_SCORES')


def validate(session):
    df_val = session.sql('''
        SELECT
            COUNT(*) AS listing_count,
            ROUND(MIN(investment_score), 2) AS min_score,
            ROUND(MAX(investment_score), 2) AS max_score,
            ROUND(AVG(investment_score), 2) AS avg_score,
            COUNT(DISTINCT neighbourhood_cleansed) AS neighbourhoods,
            COUNT(DISTINCT score_confidence) AS confidence_levels
        FROM AIRBNB_INVESTMENT_DB.GOLD.INVESTMENT_SCORES
    ''').to_pandas()
    print(df_val)

In [ ]:
# --- Orchestration ---
check_borough_summary(session)
df_silver = load_silver(session)
df_scored = compute_investment_scores(df_silver)
write_gold(session, df_scored)
validate(session)

In [ ]:
df_val = session.sql('''
    SELECT
        COUNT(*) AS listing_count,
        ROUND(MIN(investment_score), 2) AS min_score,
        ROUND(MAX(investment_score), 2) AS max_score,
        ROUND(AVG(investment_score), 2) AS avg_score,
        COUNT(DISTINCT neighbourhood_cleansed) AS neighbourhoods,
        COUNT(DISTINCT score_confidence) AS confidence_levels
    FROM AIRBNB_INVESTMENT_DB.GOLD.INVESTMENT_SCORES
''').to_pandas()
print(df_val)

## Investment Scores Complete

`GOLD.INVESTMENT_SCORES` is ready for Streamlit consumption.
Each row represents one listing with a composite investment score (0–100).
Score confidence is `medium` until transport and yield stubs are replaced with real Silver data.